# Chapter 2 — An Async-First Streaming Agent Runtime

This chapter evolves the typed Chapter 1 model seam into one asynchronous Runtime that accepts work, exposes ordered passive Events, settles failures coherently, owns classified retries, and coordinates cancellation without terminal I/O.

## Goal and Previous Limitation

Chapter 1 can assemble exactly one model stream, but callers must block inside `complete`, provider failures escape as exceptions, and there is no live handle for observation or cancellation. We first import that immutable Checkpoint and make the missing Runtime capability observable.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_1 = ROOT / 'course' / 'checkpoints' / 'ch01'
sys.path.insert(0, str(CHAPTER_1 / 'src'))
import agent_harness as chapter1

assert hasattr(chapter1, 'complete')
assert not hasattr(chapter1, 'AgentRuntime')
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]
sys.path.pop(0)


## Conceptual Model

`AgentRuntime` is the deep module. Its small interface is `start(messages)` for control and observation plus `run(messages)` for completion-oriented callers. A `ModelAdapter` still performs exactly one attempt. Runtime owns attempt state, retry classification, bounded sleeping, event sequencing, cancellation settlement, and the single conversation commit.

`AgentRunHandle.events()` is an ordered, provider-neutral observational stream. `result()` returns one `AssistantOutcome`; `cancel()` interrupts provider work or backoff. Only a successful or terminal outcome enters history, so failed partial attempts remain diagnostics.

## Minimal Execution

The next Export Cells replace the evolved model interface, add the Runtime, and publish the Chapter 2 interface. They are complete source files rather than fragments.

In [ ]:
MODEL_SOURCE = r'''"""Provider-neutral messages, model events, and model adapters."""

from __future__ import annotations

from collections.abc import AsyncIterator, Mapping, Sequence
from dataclasses import dataclass, field
from enum import Enum
import json
from types import MappingProxyType
from typing import Literal, Protocol, TypeAlias, cast
from urllib.parse import urlparse

import openai


class Role(str, Enum):
    SYSTEM = 'system'
    USER = 'user'
    ASSISTANT = 'assistant'


class StopReason(str, Enum):
    COMPLETE = 'complete'
    TOOL_USE = 'tool_use'
    LENGTH = 'length'
    CONTENT_FILTER = 'content_filter'
    ERROR = 'error'
    ABORTED = 'aborted'
    OTHER = 'other'


class ModelErrorCode(str, Enum):
    AUTHENTICATION = 'authentication'
    REQUEST = 'request'
    SCHEMA = 'schema'
    RATE_LIMIT = 'rate_limit'
    TIMEOUT = 'timeout'
    CONNECTION = 'connection'
    SERVER = 'server'
    PROVIDER = 'provider'


class UnsupportedContentError(ValueError):
    """Raised before provider I/O for a content variant outside version one."""


class ModelProtocolError(RuntimeError):
    """Raised when an adapter emits an incomplete provider-neutral stream."""


@dataclass(frozen=True, slots=True)
class TextContent:
    text: str
    type: Literal['text'] = field(default='text', init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not isinstance(self.text, str):
            raise TypeError('TextContent.text must be a string')


@dataclass(frozen=True, slots=True)
class ToolCallContent:
    id: str
    name: str
    arguments: str
    type: Literal['tool_call'] = field(default='tool_call', init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not self.id or not self.name:
            raise ValueError('a Tool Call requires non-empty id and name')
        if not isinstance(self.arguments, str):
            raise TypeError('ToolCallContent.arguments must be a JSON string')


ContentBlock: TypeAlias = TextContent | ToolCallContent


@dataclass(frozen=True, slots=True)
class AgentMessage:
    role: Role
    content: tuple[ContentBlock, ...]

    @classmethod
    def text(cls, role: Role, text: str) -> 'AgentMessage':
        return cls(role=role, content=(TextContent(text),))


@dataclass(frozen=True, slots=True)
class ModelMessage:
    role: Role
    content: tuple[ContentBlock, ...]


@dataclass(frozen=True, slots=True)
class ModelSpec:
    model_id: str
    context_window: int | None = None
    max_output_tokens: int = 4096
    supports_tools: bool = True

    def __post_init__(self) -> None:
        if not self.model_id.strip():
            raise ValueError('ModelSpec.model_id cannot be empty')
        if self.context_window is not None and self.context_window <= 0:
            raise ValueError('context_window must be positive when supplied')
        if self.max_output_tokens <= 0:
            raise ValueError('max_output_tokens must be positive')


@dataclass(frozen=True, slots=True)
class Usage:
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated: bool = False


@dataclass(frozen=True, slots=True)
class ModelRequest:
    messages: tuple[ModelMessage, ...]
    model: ModelSpec


@dataclass(frozen=True, slots=True)
class TextDelta:
    text: str


@dataclass(frozen=True, slots=True)
class ToolCallDelta:
    index: int
    id: str = ''
    name: str = ''
    arguments_delta: str = ''


@dataclass(frozen=True, slots=True)
class UsageUpdate:
    usage: Usage


@dataclass(frozen=True, slots=True)
class ModelEnd:
    stop_reason: StopReason


ModelEvent: TypeAlias = TextDelta | ToolCallDelta | UsageUpdate | ModelEnd


@dataclass(frozen=True, slots=True)
class ModelResult:
    message: ModelMessage
    stop_reason: StopReason
    usage: Usage | None = None


@dataclass(frozen=True, slots=True)
class ModelError:
    code: ModelErrorCode
    message: str
    retryable: bool
    status_code: int | None = None
    retry_after_seconds: float | None = None


class ModelAdapterError(RuntimeError):
    def __init__(self, error: ModelError) -> None:
        self.error = error
        super().__init__(error.message)


class ModelAdapter(Protocol):
    def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]: ...


def _validate_content(block: object, role: Role) -> ContentBlock:
    if not isinstance(block, (TextContent, ToolCallContent)):
        raise UnsupportedContentError(
            'version one supports only TextContent and ToolCallContent'
        )
    if getattr(block, 'schema_version', None) != 1:
        raise UnsupportedContentError('unsupported Content Block schema version')
    if isinstance(block, ToolCallContent) and role is not Role.ASSISTANT:
        raise UnsupportedContentError('ToolCallContent is valid only for assistant messages')
    return block


def to_model_messages(messages: Sequence[AgentMessage]) -> tuple[ModelMessage, ...]:
    converted: list[ModelMessage] = []
    for message in messages:
        if not isinstance(message, AgentMessage):
            raise TypeError('model input must contain AgentMessage values')
        content = tuple(_validate_content(block, message.role) for block in message.content)
        converted.append(ModelMessage(role=message.role, content=content))
    return tuple(converted)


class ScriptedModelAdapter:
    """Replay a provider-neutral event script without credentials or I/O."""

    def __init__(self, events: Sequence[ModelEvent]) -> None:
        self._events = tuple(events)
        self._requests: list[ModelRequest] = []

    @property
    def received_requests(self) -> tuple[ModelRequest, ...]:
        return tuple(self._requests)

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        self._requests.append(request)
        for event in self._events:
            yield event


async def complete(
    adapter: ModelAdapter,
    messages: Sequence[AgentMessage],
    model: ModelSpec,
) -> ModelResult:
    request = ModelRequest(to_model_messages(messages), model)
    text_parts: list[str] = []
    tool_drafts: dict[int, dict[str, str]] = {}
    usage: Usage | None = None
    end: ModelEnd | None = None
    async for event in adapter.stream(request):
        if end is not None:
            raise ModelProtocolError('an adapter emitted data after ModelEnd')
        if isinstance(event, TextDelta):
            text_parts.append(event.text)
        elif isinstance(event, ToolCallDelta):
            if event.index < 0:
                raise ModelProtocolError('Tool Call indexes cannot be negative')
            draft = tool_drafts.setdefault(
                event.index, {'id': '', 'name': '', 'arguments': ''}
            )
            draft['id'] += event.id
            draft['name'] += event.name
            draft['arguments'] += event.arguments_delta
        elif isinstance(event, UsageUpdate):
            usage = event.usage
        elif isinstance(event, ModelEnd):
            end = event
        else:
            raise ModelProtocolError(f'unsupported model event: {type(event).__name__}')
    if end is None:
        raise ModelProtocolError('an adapter stream must end with ModelEnd')
    blocks: list[ContentBlock] = []
    if text_parts:
        blocks.append(TextContent(''.join(text_parts)))
    for index in sorted(tool_drafts):
        draft = tool_drafts[index]
        try:
            blocks.append(ToolCallContent(**draft))
        except (TypeError, ValueError) as error:
            raise ModelProtocolError(f'incomplete Tool Call at index {index}') from error
    return ModelResult(
        message=ModelMessage(Role.ASSISTANT, tuple(blocks)),
        stop_reason=end.stop_reason,
        usage=usage,
    )


@dataclass(frozen=True, slots=True)
class OpenAICompatibleConfig:
    base_url: str
    api_key: str
    headers: Mapping[str, str] = field(default_factory=dict)
    extra_body: Mapping[str, object] = field(default_factory=dict)
    timeout_seconds: float = 60.0

    def __post_init__(self) -> None:
        parsed = urlparse(self.base_url)
        if parsed.scheme not in {'http', 'https'} or not parsed.netloc:
            raise ValueError('base_url must be an explicit HTTP(S) URL')
        if not self.api_key:
            raise ValueError('api_key must be supplied explicitly')
        if self.timeout_seconds <= 0:
            raise ValueError('timeout_seconds must be positive')
        object.__setattr__(self, 'headers', MappingProxyType(dict(self.headers)))
        object.__setattr__(self, 'extra_body', MappingProxyType(dict(self.extra_body)))


def _provider_message(message: ModelMessage) -> dict[str, object]:
    text = ''.join(block.text for block in message.content if isinstance(block, TextContent))
    tool_calls = [block for block in message.content if isinstance(block, ToolCallContent)]
    encoded: dict[str, object] = {'role': message.role.value, 'content': text or None}
    if tool_calls:
        encoded['tool_calls'] = [
            {
                'id': block.id,
                'type': 'function',
                'function': {'name': block.name, 'arguments': block.arguments},
            }
            for block in tool_calls
        ]
    return encoded


def _stop_reason(value: str | None) -> StopReason:
    if value is None:
        return StopReason.OTHER
    return {
        'stop': StopReason.COMPLETE,
        'tool_calls': StopReason.TOOL_USE,
        'length': StopReason.LENGTH,
        'content_filter': StopReason.CONTENT_FILTER,
    }.get(value, StopReason.OTHER)


def _normalized_error(error: Exception) -> ModelError:
    status = getattr(error, 'status_code', None)
    retry_after_seconds: float | None = None
    response = getattr(error, 'response', None)
    headers = getattr(response, 'headers', None)
    if headers is not None:
        raw_retry_after = headers.get('retry-after')
        if raw_retry_after is not None:
            try:
                parsed_retry_after = float(raw_retry_after)
            except (TypeError, ValueError):
                pass
            else:
                if parsed_retry_after >= 0:
                    retry_after_seconds = parsed_retry_after
    if isinstance(error, openai.AuthenticationError):
        code, retryable = ModelErrorCode.AUTHENTICATION, False
    elif isinstance(error, openai.RateLimitError) or status == 429:
        code, retryable = ModelErrorCode.RATE_LIMIT, True
    elif isinstance(error, openai.APITimeoutError) or status == 408:
        code, retryable = ModelErrorCode.TIMEOUT, True
    elif isinstance(error, openai.APIConnectionError):
        code, retryable = ModelErrorCode.CONNECTION, True
    elif isinstance(status, int) and status >= 500:
        code, retryable = ModelErrorCode.SERVER, True
    elif isinstance(error, (openai.BadRequestError, openai.NotFoundError)):
        code, retryable = ModelErrorCode.REQUEST, False
    else:
        code, retryable = ModelErrorCode.PROVIDER, False
    status_text = f' with status {status}' if status is not None else ''
    return ModelError(
        code=code,
        message=f'OpenAI-compatible request failed{status_text}',
        retryable=retryable,
        status_code=status,
        retry_after_seconds=retry_after_seconds,
    )


class OpenAICompatibleAdapter:
    """Translate the streaming Chat Completions protocol at one seam."""

    def __init__(self, config: OpenAICompatibleConfig) -> None:
        self._config = config
        self._client = openai.AsyncOpenAI(
            api_key=config.api_key,
            base_url=config.base_url.rstrip('/') + '/',
            default_headers=dict(config.headers),
            timeout=config.timeout_seconds,
            max_retries=0,
        )

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        finish_reason: str | None = None
        try:
            response = await self._client.chat.completions.create(
                model=request.model.model_id,
                messages=cast(list, [_provider_message(item) for item in request.messages]),
                max_tokens=request.model.max_output_tokens,
                stream=True,
                stream_options={'include_usage': True},
                extra_body=dict(self._config.extra_body) or None,
            )
            async for chunk in response:
                if chunk.usage is not None:
                    input_tokens = chunk.usage.prompt_tokens or 0
                    output_tokens = chunk.usage.completion_tokens or 0
                    total_tokens = chunk.usage.total_tokens or input_tokens + output_tokens
                    yield UsageUpdate(Usage(input_tokens, output_tokens, total_tokens))
                for choice in chunk.choices:
                    delta = choice.delta
                    if delta.content:
                        yield TextDelta(delta.content)
                    for tool_call in delta.tool_calls or ():
                        function = tool_call.function
                        yield ToolCallDelta(
                            index=tool_call.index,
                            id=tool_call.id or '',
                            name=(function.name if function else None) or '',
                            arguments_delta=(function.arguments if function else None) or '',
                        )
                    if choice.finish_reason is not None:
                        finish_reason = choice.finish_reason
        except openai.OpenAIError as error:
            raise ModelAdapterError(_normalized_error(error)) from None
        yield ModelEnd(_stop_reason(finish_reason))
'''


In [ ]:
RUNTIME_SOURCE = r'''"""Async-first agent execution with ordered passive observations."""

from __future__ import annotations

import asyncio
from collections.abc import AsyncIterator, Awaitable, Callable, Sequence
from dataclasses import dataclass
from enum import Enum
from typing import TypeAlias, cast

from .model import (
    AgentMessage,
    ContentBlock,
    ModelAdapter,
    ModelAdapterError,
    ModelEnd,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelRequest,
    ModelSpec,
    Role,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    Usage,
    UsageUpdate,
    to_model_messages,
)


class EventType(str, Enum):
    AGENT_START = "agent_start"
    MODEL_ATTEMPT_START = "model_attempt_start"
    MODEL_EVENT = "model_event"
    MODEL_ATTEMPT_FAILED = "model_attempt_failed"
    RETRY_SCHEDULED = "retry_scheduled"
    RUN_CANCELLED = "run_cancelled"
    MESSAGE_END = "message_end"
    AGENT_END = "agent_end"


@dataclass(frozen=True, slots=True)
class RuntimeEvent:
    sequence: int
    type: EventType
    attempt: int | None = None
    model_event: ModelEvent | None = None
    error: ModelError | None = None
    retry_delay_seconds: float | None = None
    partial_text: str = ""
    partial_usage: Usage | None = None


@dataclass(frozen=True, slots=True)
class AssistantOutcome:
    message: AgentMessage
    stop_reason: StopReason
    usage: Usage | None = None
    error: ModelError | None = None
    attempts: int = 1


@dataclass(frozen=True, slots=True)
class RetryPolicy:
    delays: tuple[float, ...] = (2.0, 4.0, 8.0)
    max_retry_after_seconds: float = 60.0

    def __post_init__(self) -> None:
        if any(delay < 0 for delay in self.delays):
            raise ValueError("retry delays cannot be negative")
        if self.max_retry_after_seconds < 0:
            raise ValueError("max_retry_after_seconds cannot be negative")

    def delay_for(
        self,
        error: ModelError,
        failed_attempt: int,
        *,
        retry_after_seconds: float | None = None,
    ) -> float | None:
        retryable_codes = {
            ModelErrorCode.RATE_LIMIT,
            ModelErrorCode.TIMEOUT,
            ModelErrorCode.CONNECTION,
            ModelErrorCode.SERVER,
        }
        retryable_status = error.status_code in {408, 429} or (
            error.status_code is not None and error.status_code >= 500
        )
        if (
            error.code not in retryable_codes
            and not retryable_status
        ) or failed_attempt > len(self.delays):
            return None
        if retry_after_seconds is not None:
            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:
                return None
            return retry_after_seconds
        return self.delays[failed_attempt - 1]


Sleeper: TypeAlias = Callable[[float], Awaitable[None]]
_EVENTS_DONE = object()


class AgentRunHandle:
    """One accepted run's observations, cancellation, and eventual outcome."""

    def __init__(
        self,
        task: asyncio.Task[AssistantOutcome],
        events: asyncio.Queue[RuntimeEvent | object],
    ) -> None:
        self._task = task
        self._events = events
        self._cancel_requested = False

    async def events(self) -> AsyncIterator[RuntimeEvent]:
        while True:
            event = await self._events.get()
            if event is _EVENTS_DONE:
                break
            yield cast(RuntimeEvent, event)

    async def result(self) -> AssistantOutcome:
        return await self._task

    def cancel(self) -> None:
        if not self._cancel_requested and not self._task.done():
            self._cancel_requested = True
            self._task.get_loop().call_soon(self._task.cancel)


class AgentRuntime:
    """Advance typed conversation state through one async model runtime."""

    def __init__(
        self,
        adapter: ModelAdapter,
        model: ModelSpec,
        *,
        retry_policy: RetryPolicy | None = None,
        sleeper: Sleeper = asyncio.sleep,
        run_guard: object | None = None,
    ) -> None:
        if not isinstance(model, ModelSpec):
            raise TypeError("model must be a ModelSpec")
        if not callable(getattr(adapter, "stream", None)):
            raise TypeError("adapter must implement ModelAdapter.stream")
        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):
            raise TypeError("retry_policy must be a RetryPolicy")
        if not callable(sleeper):
            raise TypeError("sleeper must be an async callable")
        self._adapter = adapter
        self._model = model
        self._retry_policy = retry_policy or RetryPolicy()
        self._sleeper = sleeper
        self._run_guard = run_guard
        self._history: list[AgentMessage] = []

    @property
    def history(self) -> tuple[AgentMessage, ...]:
        return tuple(self._history)

    @property
    def run_guard(self) -> object | None:
        return self._run_guard

    def start(self, messages: Sequence[AgentMessage]) -> AgentRunHandle:
        loop = asyncio.get_running_loop()
        accepted = tuple(messages)
        request = ModelRequest(
            to_model_messages((*self._history, *accepted)), self._model
        )
        self._history.extend(accepted)
        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()
        task = loop.create_task(self._execute(request, events))
        return AgentRunHandle(task, events)

    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:
        return await self.start(messages).result()

    async def _execute(
        self,
        request: ModelRequest,
        events: asyncio.Queue[RuntimeEvent | object],
    ) -> AssistantOutcome:
        sequence = 0
        attempt = 0
        text_parts: list[str] = []
        usage: Usage | None = None

        async def emit(
            type_: EventType,
            *,
            attempt: int | None = None,
            model_event: ModelEvent | None = None,
            error: ModelError | None = None,
            retry_delay_seconds: float | None = None,
            partial_text: str = "",
            partial_usage: Usage | None = None,
        ) -> None:
            nonlocal sequence
            sequence += 1
            await events.put(
                RuntimeEvent(
                    sequence=sequence,
                    type=type_,
                    attempt=attempt,
                    model_event=model_event,
                    error=error,
                    retry_delay_seconds=retry_delay_seconds,
                    partial_text=partial_text,
                    partial_usage=partial_usage,
                )
            )

        try:
            await emit(EventType.AGENT_START)
            while True:
                attempt += 1
                await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)
                text_parts = []
                tool_drafts: dict[int, dict[str, str]] = {}
                usage = None
                end: ModelEnd | None = None
                schema_error: ModelError | None = None
                try:
                    async for event in self._adapter.stream(request):
                        await emit(
                            EventType.MODEL_EVENT,
                            attempt=attempt,
                            model_event=event,
                        )
                        if end is not None:
                            schema_error = ModelError(
                                ModelErrorCode.SCHEMA,
                                "model stream emitted data after ModelEnd",
                                False,
                            )
                            break
                        if isinstance(event, TextDelta):
                            text_parts.append(event.text)
                        elif isinstance(event, ToolCallDelta):
                            if event.index < 0:
                                schema_error = ModelError(
                                    ModelErrorCode.SCHEMA,
                                    "model stream emitted an invalid Tool Call index",
                                    False,
                                )
                                break
                            draft = tool_drafts.setdefault(
                                event.index,
                                {"id": "", "name": "", "arguments": ""},
                            )
                            draft["id"] += event.id
                            draft["name"] += event.name
                            draft["arguments"] += event.arguments_delta
                        elif isinstance(event, UsageUpdate):
                            usage = event.usage
                        elif isinstance(event, ModelEnd):
                            end = event
                        else:
                            schema_error = ModelError(
                                ModelErrorCode.SCHEMA,
                                "model stream emitted an unsupported event",
                                False,
                            )
                            break
                except ModelAdapterError as failure:
                    partial_text = "".join(text_parts)
                    await emit(
                        EventType.MODEL_ATTEMPT_FAILED,
                        attempt=attempt,
                        error=failure.error,
                        partial_text=partial_text,
                        partial_usage=usage,
                    )
                    delay = self._retry_policy.delay_for(
                        failure.error,
                        attempt,
                        retry_after_seconds=failure.error.retry_after_seconds,
                    )
                    if delay is not None:
                        await emit(
                            EventType.RETRY_SCHEDULED,
                            attempt=attempt,
                            error=failure.error,
                            retry_delay_seconds=delay,
                            partial_text=partial_text,
                            partial_usage=usage,
                        )
                        text_parts = []
                        usage = None
                        await self._sleeper(delay)
                        continue
                    message = AgentMessage.text(Role.ASSISTANT, partial_text)
                    outcome = AssistantOutcome(
                        message=message,
                        stop_reason=StopReason.ERROR,
                        usage=usage,
                        error=failure.error,
                        attempts=attempt,
                    )
                else:
                    error = schema_error
                    if error is None and end is None:
                        error = ModelError(
                            ModelErrorCode.SCHEMA,
                            "model stream violated the provider-neutral event contract",
                            False,
                        )
                    if error is None:
                        try:
                            blocks: list[ContentBlock] = []
                            if text_parts:
                                blocks.append(TextContent("".join(text_parts)))
                            for index in sorted(tool_drafts):
                                blocks.append(ToolCallContent(**tool_drafts[index]))
                        except (TypeError, ValueError):
                            error = ModelError(
                                ModelErrorCode.SCHEMA,
                                "model stream emitted an incomplete Tool Call",
                                False,
                            )
                    if error is not None:
                        partial_text = "".join(text_parts)
                        await emit(
                            EventType.MODEL_ATTEMPT_FAILED,
                            attempt=attempt,
                            error=error,
                            partial_text=partial_text,
                            partial_usage=usage,
                        )
                        outcome = AssistantOutcome(
                            message=AgentMessage.text(Role.ASSISTANT, partial_text),
                            stop_reason=StopReason.ERROR,
                            usage=usage,
                            error=error,
                            attempts=attempt,
                        )
                    else:
                        assert end is not None
                        message = AgentMessage(Role.ASSISTANT, tuple(blocks))
                        outcome = AssistantOutcome(
                            message, end.stop_reason, usage, attempts=attempt
                        )
                self._history.append(outcome.message)
                await emit(EventType.MESSAGE_END, attempt=attempt)
                await emit(EventType.AGENT_END, attempt=attempt)
                return outcome
        except asyncio.CancelledError:
            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))
            outcome = AssistantOutcome(
                message=message,
                stop_reason=StopReason.ABORTED,
                usage=usage,
                attempts=max(attempt, 1),
            )
            self._history.append(message)
            await emit(
                EventType.RUN_CANCELLED,
                attempt=max(attempt, 1),
                partial_text="".join(text_parts),
                partial_usage=usage,
            )
            await emit(EventType.MESSAGE_END, attempt=max(attempt, 1))
            await emit(EventType.AGENT_END, attempt=max(attempt, 1))
            return outcome
        finally:
            await events.put(_EVENTS_DONE)
'''


In [ ]:
INIT_SOURCE = r'''"""Public interface for the Chapter 2 Agent Harness Checkpoint."""

from .model import (
    AgentMessage,
    ContentBlock,
    ModelAdapter,
    ModelAdapterError,
    ModelEnd,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelMessage,
    ModelProtocolError,
    ModelRequest,
    ModelResult,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    ScriptedModelAdapter,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    UnsupportedContentError,
    Usage,
    UsageUpdate,
    complete,
    to_model_messages,
)
from .runtime import (
    AgentRunHandle,
    AgentRuntime,
    AssistantOutcome,
    EventType,
    RetryPolicy,
    RuntimeEvent,
    Sleeper,
)

__all__ = [name for name in globals() if not name.startswith('_')]
'''


In [ ]:
import asyncio
import importlib
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='chapter-02-minimal-') as temporary:
    package = Path(temporary) / 'agent_harness'
    package.mkdir()
    (package / 'model.py').write_text(MODEL_SOURCE, encoding='utf-8')
    (package / 'runtime.py').write_text(RUNTIME_SOURCE, encoding='utf-8')
    (package / '__init__.py').write_text(INIT_SOURCE, encoding='utf-8')
    sys.path.insert(0, temporary)
    try:
        chapter2 = importlib.import_module('agent_harness')

        async def minimal_run():
            adapter = chapter2.ScriptedModelAdapter([
                chapter2.TextDelta('async '),
                chapter2.TextDelta('answer'),
                chapter2.UsageUpdate(chapter2.Usage(2, 2, 4)),
                chapter2.ModelEnd(chapter2.StopReason.COMPLETE),
            ])
            runtime = chapter2.AgentRuntime(
                adapter, chapter2.ModelSpec('scripted/chapter-02')
            )
            handle = runtime.start([
                chapter2.AgentMessage.text(chapter2.Role.USER, 'stream it')
            ])
            observed = [event async for event in handle.events()]
            return await handle.result(), observed, runtime.history

        minimal_outcome, minimal_events, minimal_history = await minimal_run()
    finally:
        sys.path.pop(0)
        for module_name in tuple(sys.modules):
            if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
                del sys.modules[module_name]

assert minimal_outcome.message.content[0].text == 'async answer'
assert minimal_events[0].type.value == 'agent_start'
assert minimal_events[-1].type.value == 'agent_end'
assert minimal_history[-1] == minimal_outcome.message


## Staged Construction

The tracer bullet first proved `start` could stream ordered Events and settle a typed outcome. Successive red–green cycles added terminal partial failure, the 2/4/8 retry schedule, bounded `Retry-After`, Runtime-owned classification, Schema settlement, provider and backoff cancellation, pre-acceptance validation, opt-in guards, and immediate-cancel race handling.

The retry sleeper is injected, so deterministic tests never wait in wall-clock time. OpenAI-compatible transport remains explicit and configures SDK retries off; it surfaces normalized errors and retry hints for Runtime policy.

In [ ]:
RUNTIME_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
from collections.abc import AsyncIterator
from dataclasses import dataclass

import pytest

from agent_harness import (
    AgentMessage,
    AgentRuntime,
    EventType,
    ModelEnd,
    ModelAdapterError,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelRequest,
    ModelMessage,
    ModelSpec,
    Role,
    RetryPolicy,
    ScriptedModelAdapter,
    StopReason,
    TextContent,
    TextDelta,
    Usage,
    UsageUpdate,
    UnsupportedContentError,
)


def test_started_run_streams_ordered_events_and_settles_typed_outcome() -> None:
    async def scenario() -> None:
        prompt = AgentMessage.text(Role.USER, "hello runtime")
        adapter = ScriptedModelAdapter(
            [
                TextDelta("streamed "),
                TextDelta("answer"),
                UsageUpdate(Usage(3, 2, 5)),
                ModelEnd(StopReason.COMPLETE),
            ]
        )
        runtime = AgentRuntime(adapter, ModelSpec("scripted/runtime"))

        handle = runtime.start([prompt])
        events = [event async for event in handle.events()]
        outcome = await handle.result()

        assert [event.sequence for event in events] == list(range(1, len(events) + 1))
        assert [event.type for event in events] == [
            EventType.AGENT_START,
            EventType.MODEL_ATTEMPT_START,
            EventType.MODEL_EVENT,
            EventType.MODEL_EVENT,
            EventType.MODEL_EVENT,
            EventType.MODEL_EVENT,
            EventType.MESSAGE_END,
            EventType.AGENT_END,
        ]
        assert outcome.message == AgentMessage(
            Role.ASSISTANT, (TextContent("streamed answer"),)
        )
        assert outcome.stop_reason is StopReason.COMPLETE
        assert outcome.usage == Usage(3, 2, 5)
        assert outcome.error is None
        assert runtime.history == (prompt, outcome.message)
        assert adapter.received_requests[0].messages == (
            ModelMessage(Role.USER, (TextContent("hello runtime"),)),
        )

    asyncio.run(scenario())


def test_provider_failure_after_acceptance_settles_partial_assistant_outcome() -> None:
    class FailingAdapter:
        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            yield TextDelta("partial answer")
            yield UsageUpdate(Usage(7, 2, 9))
            raise ModelAdapterError(
                ModelError(
                    ModelErrorCode.SERVER,
                    "OpenAI-compatible request failed with status 503",
                    retryable=True,
                    status_code=503,
                )
            )

    async def scenario() -> None:
        prompt = AgentMessage.text(Role.USER, "fail coherently")
        runtime = AgentRuntime(
            FailingAdapter(),
            ModelSpec("scripted/failure"),
            retry_policy=RetryPolicy(delays=()),
        )

        handle = runtime.start([prompt])
        events = [event async for event in handle.events()]
        outcome = await handle.result()

        assert outcome.stop_reason is StopReason.ERROR
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "partial answer")
        assert outcome.usage == Usage(7, 2, 9)
        assert outcome.error == ModelError(
            ModelErrorCode.SERVER,
            "OpenAI-compatible request failed with status 503",
            retryable=True,
            status_code=503,
        )
        failure = next(
            event for event in events if event.type is EventType.MODEL_ATTEMPT_FAILED
        )
        assert failure.partial_text == "partial answer"
        assert failure.partial_usage == Usage(7, 2, 9)
        assert events[-2].type is EventType.MESSAGE_END
        assert events[-1].type is EventType.AGENT_END
        assert runtime.history == (prompt, outcome.message)

    asyncio.run(scenario())


def test_transient_failures_retry_on_the_default_two_four_eight_schedule() -> None:
    class AttemptAdapter:
        def __init__(self) -> None:
            self.attempt = 0

        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            self.attempt += 1
            if self.attempt <= 3:
                yield TextDelta(f"discard attempt {self.attempt}")
                code, status = (
                    (ModelErrorCode.TIMEOUT, 408),
                    (ModelErrorCode.RATE_LIMIT, 429),
                    (ModelErrorCode.SERVER, 503),
                )[self.attempt - 1]
                raise ModelAdapterError(
                    ModelError(code, f"transient {status}", True, status)
                )
            yield TextDelta("final answer")
            yield ModelEnd(StopReason.COMPLETE)

    async def scenario() -> None:
        delays: list[float] = []

        async def record_sleep(delay: float) -> None:
            delays.append(delay)

        adapter = AttemptAdapter()
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/retries"),
            sleeper=record_sleep,
        )

        handle = runtime.start([AgentMessage.text(Role.USER, "retry safely")])
        events = [event async for event in handle.events()]
        outcome = await handle.result()

        assert delays == [2.0, 4.0, 8.0]
        assert adapter.attempt == 4
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "final answer")
        assert outcome.attempts == 4
        assert [
            event.partial_text
            for event in events
            if event.type is EventType.MODEL_ATTEMPT_FAILED
        ] == ["discard attempt 1", "discard attempt 2", "discard attempt 3"]
        assert [
            message
            for message in runtime.history
            if message.role is Role.ASSISTANT
        ] == [outcome.message]

    asyncio.run(scenario())


def test_retry_after_is_honored_only_within_the_bounded_policy() -> None:
    class RetryAfterAdapter:
        def __init__(self, retry_after: float) -> None:
            self.retry_after = retry_after
            self.attempt = 0

        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            self.attempt += 1
            if self.attempt == 1:
                raise ModelAdapterError(
                    ModelError(
                        ModelErrorCode.RATE_LIMIT,
                        "rate limited",
                        True,
                        429,
                        self.retry_after,
                    )
                )
            yield TextDelta("recovered")
            yield ModelEnd(StopReason.COMPLETE)

    async def run_case(retry_after: float) -> tuple[list[float], int, AssistantOutcome]:
        delays: list[float] = []

        async def record_sleep(delay: float) -> None:
            delays.append(delay)

        adapter = RetryAfterAdapter(retry_after)
        outcome = await AgentRuntime(
            adapter,
            ModelSpec("scripted/retry-after"),
            sleeper=record_sleep,
        ).run([AgentMessage.text(Role.USER, "respect server hint")])
        return delays, adapter.attempt, outcome

    async def scenario() -> None:
        bounded = await run_case(12.5)
        excessive = await run_case(60.1)

        assert bounded[:2] == ([12.5], 2)
        assert bounded[2].stop_reason is StopReason.COMPLETE
        assert excessive[:2] == ([], 1)
        assert excessive[2].stop_reason is StopReason.ERROR
        assert excessive[2].error is not None
        assert excessive[2].error.retry_after_seconds == 60.1

    asyncio.run(scenario())


def test_runtime_classifies_retryability_from_normalized_error_codes() -> None:
    class ClassifiedAdapter:
        def __init__(self, error: ModelError) -> None:
            self.error = error
            self.attempt = 0

        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            self.attempt += 1
            if self.attempt == 1:
                raise ModelAdapterError(self.error)
            yield TextDelta("recovered")
            yield ModelEnd(StopReason.COMPLETE)

    async def run_case(error: ModelError) -> tuple[int, list[float], AssistantOutcome]:
        delays: list[float] = []

        async def record_sleep(delay: float) -> None:
            delays.append(delay)

        adapter = ClassifiedAdapter(error)
        outcome = await AgentRuntime(
            adapter,
            ModelSpec("scripted/classification"),
            sleeper=record_sleep,
        ).run([AgentMessage.text(Role.USER, "classify")])
        return adapter.attempt, delays, outcome

    async def scenario() -> None:
        connection = await run_case(
            ModelError(ModelErrorCode.CONNECTION, "connection failed", False)
        )
        authentication = await run_case(
            ModelError(ModelErrorCode.AUTHENTICATION, "bad credential", True, 401)
        )
        request = await run_case(
            ModelError(ModelErrorCode.REQUEST, "invalid request", True, 400)
        )

        assert connection[:2] == (2, [2.0])
        assert connection[2].stop_reason is StopReason.COMPLETE
        assert authentication[:2] == (1, [])
        assert authentication[2].stop_reason is StopReason.ERROR
        assert request[:2] == (1, [])
        assert request[2].stop_reason is StopReason.ERROR

    asyncio.run(scenario())


def test_stream_schema_failure_settles_without_retrying_or_leaking_exception() -> None:
    async def scenario() -> None:
        delays: list[float] = []

        async def record_sleep(delay: float) -> None:
            delays.append(delay)

        runtime = AgentRuntime(
            ScriptedModelAdapter([TextDelta("incomplete")]),
            ModelSpec("scripted/schema-failure"),
            sleeper=record_sleep,
        )

        outcome = await runtime.run(
            [AgentMessage.text(Role.USER, "settle malformed stream")]
        )

        assert outcome.stop_reason is StopReason.ERROR
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "incomplete")
        assert outcome.error is not None
        assert outcome.error.code is ModelErrorCode.SCHEMA
        assert outcome.error.retryable is False
        assert outcome.attempts == 1
        assert delays == []

    asyncio.run(scenario())


def test_cancellation_settles_partial_provider_work_as_aborted_outcome() -> None:
    class BlockingAdapter:
        def __init__(self) -> None:
            self.started = asyncio.Event()

        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            yield TextDelta("before cancel")
            yield UsageUpdate(Usage(4, 2, 6))
            self.started.set()
            await asyncio.Event().wait()
            yield ModelEnd(StopReason.COMPLETE)

    async def scenario() -> None:
        adapter = BlockingAdapter()
        prompt = AgentMessage.text(Role.USER, "cancel this")
        runtime = AgentRuntime(adapter, ModelSpec("scripted/cancel"))
        handle = runtime.start([prompt])
        await adapter.started.wait()

        handle.cancel()
        events = [event async for event in handle.events()]
        outcome = await handle.result()

        assert outcome.stop_reason is StopReason.ABORTED
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "before cancel")
        assert outcome.usage == Usage(4, 2, 6)
        assert outcome.error is None
        assert EventType.RUN_CANCELLED in [event.type for event in events]
        assert events[-2].type is EventType.MESSAGE_END
        assert events[-1].type is EventType.AGENT_END
        assert runtime.history == (prompt, outcome.message)

    asyncio.run(scenario())


def test_cancellation_interrupts_retry_wait_without_committing_failed_partial() -> None:
    class TransientAdapter:
        def __init__(self) -> None:
            self.attempt = 0

        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            self.attempt += 1
            yield TextDelta("diagnostic only")
            raise ModelAdapterError(
                ModelError(ModelErrorCode.CONNECTION, "connection failed", True)
            )

    async def scenario() -> None:
        sleeping = asyncio.Event()

        async def blocking_sleep(delay: float) -> None:
            sleeping.set()
            await asyncio.Event().wait()

        adapter = TransientAdapter()
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/cancel-backoff"),
            sleeper=blocking_sleep,
        )
        handle = runtime.start([AgentMessage.text(Role.USER, "cancel retry")])
        await sleeping.wait()

        handle.cancel()
        events = [event async for event in handle.events()]
        outcome = await handle.result()

        assert adapter.attempt == 1
        assert outcome.stop_reason is StopReason.ABORTED
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "")
        failure = next(
            event for event in events if event.type is EventType.MODEL_ATTEMPT_FAILED
        )
        assert failure.partial_text == "diagnostic only"
        assert runtime.history[-1] == outcome.message

    asyncio.run(scenario())


def test_invalid_input_fails_before_run_acceptance_without_mutating_history() -> None:
    @dataclass(frozen=True)
    class ImageContent:
        source: str = "unsupported"
        schema_version: int = 1

    async def scenario() -> None:
        adapter = ScriptedModelAdapter([ModelEnd(StopReason.COMPLETE)])
        runtime = AgentRuntime(adapter, ModelSpec("scripted/preflight"))
        invalid = AgentMessage(Role.USER, (ImageContent(),))  # type: ignore[arg-type]

        with pytest.raises(UnsupportedContentError):
            runtime.start([invalid])

        assert runtime.history == ()
        assert adapter.received_requests == ()

    asyncio.run(scenario())


def test_aggregate_run_guard_is_opt_in() -> None:
    adapter = ScriptedModelAdapter([ModelEnd(StopReason.COMPLETE)])

    unbounded = AgentRuntime(adapter, ModelSpec("scripted/unbounded"))
    explicit_guard = object()
    bounded = AgentRuntime(
        adapter,
        ModelSpec("scripted/bounded"),
        run_guard=explicit_guard,
    )

    assert unbounded.run_guard is None
    assert bounded.run_guard is explicit_guard


def test_immediate_cancellation_still_reaches_a_settled_boundary() -> None:
    class NeverStartedAdapter:
        async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
            await asyncio.Event().wait()
            yield ModelEnd(StopReason.COMPLETE)

    async def scenario() -> None:
        runtime = AgentRuntime(
            NeverStartedAdapter(), ModelSpec("scripted/immediate-cancel")
        )
        handle = runtime.start([AgentMessage.text(Role.USER, "cancel immediately")])

        handle.cancel()
        outcome = await asyncio.wait_for(handle.result(), timeout=1)
        events = [event async for event in handle.events()]

        assert outcome.stop_reason is StopReason.ABORTED
        assert events[-1].type is EventType.AGENT_END
        assert runtime.history[-1] == outcome.message

    asyncio.run(scenario())


def test_data_after_model_end_is_a_non_retryable_schema_outcome() -> None:
    async def scenario() -> None:
        runtime = AgentRuntime(
            ScriptedModelAdapter(
                [ModelEnd(StopReason.COMPLETE), TextDelta("invalid late data")]
            ),
            ModelSpec("scripted/late-data"),
        )

        outcome = await runtime.run(
            [AgentMessage.text(Role.USER, "validate event order")]
        )

        assert outcome.stop_reason is StopReason.ERROR
        assert outcome.error is not None
        assert outcome.error.code is ModelErrorCode.SCHEMA
        assert outcome.attempts == 1

    asyncio.run(scenario())


def test_invalid_runtime_dependencies_fail_during_construction() -> None:
    adapter = ScriptedModelAdapter([ModelEnd(StopReason.COMPLETE)])
    model = ModelSpec("scripted/invalid-construction")

    with pytest.raises(TypeError, match="retry_policy"):
        AgentRuntime(adapter, model, retry_policy="invalid")  # type: ignore[arg-type]
    with pytest.raises(TypeError, match="sleeper"):
        AgentRuntime(adapter, model, sleeper=None)  # type: ignore[arg-type]

    assert adapter.received_requests == ()
'''


In [ ]:
OPENAI_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
from contextlib import contextmanager
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import json
from threading import Thread
from typing import Iterator

import pytest

from agent_harness import (
    AgentMessage,
    ModelAdapterError,
    ModelErrorCode,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    StopReason,
    TextContent,
    ToolCallContent,
    Usage,
    complete,
)


def _chunk(delta, *, finish_reason=None):
    return {
        'id': 'chatcmpl-local',
        'object': 'chat.completion.chunk',
        'created': 1,
        'model': 'local-test',
        'choices': [
            {'index': 0, 'delta': delta, 'finish_reason': finish_reason}
        ],
    }


@contextmanager
def fake_openai_server() -> Iterator[tuple[str, list[dict[str, object]]]]:
    requests: list[dict[str, object]] = []

    class Handler(BaseHTTPRequestHandler):
        def do_POST(self):
            length = int(self.headers.get('content-length', '0'))
            body = json.loads(self.rfile.read(length))
            requests.append(
                {'path': self.path, 'authorization': self.headers.get('authorization'), 'body': body}
            )
            if body['model'] in {'reject-me', 'retry-me'}:
                payload = json.dumps(
                    {'error': {'message': 'SECRET provider detail', 'type': 'invalid_request_error'}}
                ).encode()
                status = 429 if body['model'] == 'retry-me' else 401
                self.send_response(status)
                self.send_header('content-type', 'application/json')
                self.send_header('content-length', str(len(payload)))
                if status == 429:
                    self.send_header('retry-after', '7')
                self.end_headers()
                self.wfile.write(payload)
                return

            events = [
                _chunk({'role': 'assistant', 'content': 'Hel'}),
                _chunk({'content': 'lo'}),
                _chunk(
                    {
                        'tool_calls': [
                            {
                                'index': 0,
                                'id': 'call-1',
                                'type': 'function',
                                'function': {'name': 'read', 'arguments': '{"path":'},
                            }
                        ]
                    }
                ),
                _chunk(
                    {'tool_calls': [{'index': 0, 'function': {'arguments': '"README.md"}'}}]}
                ),
                _chunk({}, finish_reason='tool_calls'),
                {
                    'id': 'chatcmpl-local',
                    'object': 'chat.completion.chunk',
                    'created': 1,
                    'model': 'local-test',
                    'choices': [],
                    'usage': {'prompt_tokens': 4, 'completion_tokens': 6, 'total_tokens': 10},
                },
            ]
            payload = ''.join(f'data: {json.dumps(event)}\n\n' for event in events)
            payload += 'data: [DONE]\n\n'
            encoded = payload.encode()
            self.send_response(200)
            self.send_header('content-type', 'text/event-stream')
            self.send_header('content-length', str(len(encoded)))
            self.end_headers()
            self.wfile.write(encoded)

        def log_message(self, format, *args):
            return

    server = ThreadingHTTPServer(('127.0.0.1', 0), Handler)
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        host, port = server.server_address
        yield f'http://{host}:{port}/v1', requests
    finally:
        server.shutdown()
        server.server_close()
        thread.join(timeout=5)


def test_openai_compatible_stream_is_translated_at_the_adapter_seam(monkeypatch):
    monkeypatch.setenv('OPENAI_API_KEY', 'ambient-key-must-not-win')
    with fake_openai_server() as (base_url, requests):
        adapter = OpenAICompatibleAdapter(
            OpenAICompatibleConfig(base_url=base_url, api_key='explicit-key')
        )
        result = asyncio.run(
            complete(
                adapter,
                [AgentMessage.text(Role.USER, 'inspect')],
                ModelSpec('local-test', max_output_tokens=128),
            )
        )

    assert result.message.content == (
        TextContent('Hello'),
        ToolCallContent('call-1', 'read', '{"path":"README.md"}'),
    )
    assert result.stop_reason is StopReason.TOOL_USE
    assert result.usage == Usage(4, 6, 10)
    assert requests[0]['path'] == '/v1/chat/completions'
    assert requests[0]['authorization'] == 'Bearer explicit-key'
    assert requests[0]['body']['messages'] == [{'role': 'user', 'content': 'inspect'}]


def test_openai_compatible_failure_is_normalized_without_raw_detail():
    with fake_openai_server() as (base_url, _):
        adapter = OpenAICompatibleAdapter(
            OpenAICompatibleConfig(base_url=base_url, api_key='explicit-key')
        )
        with pytest.raises(ModelAdapterError) as captured:
            asyncio.run(
                complete(
                    adapter,
                    [AgentMessage.text(Role.USER, 'fail safely')],
                    ModelSpec('reject-me'),
                )
            )

    assert captured.value.error.code is ModelErrorCode.AUTHENTICATION
    assert captured.value.error.status_code == 401
    assert captured.value.error.retryable is False
    assert 'SECRET' not in str(captured.value)


def test_adapter_surfaces_bounded_retry_hint_without_retrying_itself():
    with fake_openai_server() as (base_url, requests):
        adapter = OpenAICompatibleAdapter(
            OpenAICompatibleConfig(base_url=base_url, api_key='explicit-key')
        )
        with pytest.raises(ModelAdapterError) as captured:
            asyncio.run(
                complete(
                    adapter,
                    [AgentMessage.text(Role.USER, 'retry in runtime')],
                    ModelSpec('retry-me'),
                )
            )

    assert captured.value.error.code is ModelErrorCode.RATE_LIMIT
    assert captured.value.error.retry_after_seconds == 7.0
    assert len(requests) == 1
'''


## Observable Trace

Events carry a monotonic sequence, attempt number, provider-neutral model event, normalized error, retry delay, and bounded partial-attempt evidence as appropriate. The trace below comes from the same minimal execution and requires no renderer or terminal.

In [ ]:
observable_trace = [
    {
        'sequence': event.sequence,
        'type': event.type.value,
        'attempt': event.attempt,
    }
    for event in minimal_events
]
assert [item['sequence'] for item in observable_trace] == list(
    range(1, len(observable_trace) + 1)
)
observable_trace


## Failure Boundaries and Trade-offs

Invalid configuration or unsupported input is rejected synchronously before acceptance and cannot mutate history. After acceptance, provider and stream-Schema failures settle as `StopReason.ERROR`; cancellation settles as `StopReason.ABORTED`. The final failed attempt may contribute partial text and usage to its terminal outcome, while retried partial attempts remain Events only.

Runtime retries connection, timeout/408, rate-limit/429, and 5xx failures. Authentication, request, Schema, cancellation, and unknown provider failures do not retry. `Retry-After` above 60 seconds fails immediately and visibly. Aggregate turn, Tool-call, duration, and token guards remain absent unless a caller explicitly supplies one.

In [ ]:
schema_outcome = await (
    chapter2.AgentRuntime(
        chapter2.ScriptedModelAdapter([chapter2.TextDelta('partial')]),
        chapter2.ModelSpec('scripted/schema-demo'),
    ).run([chapter2.AgentMessage.text(chapter2.Role.USER, 'malformed')])
)
assert schema_outcome.stop_reason is chapter2.StopReason.ERROR
assert schema_outcome.error.code is chapter2.ModelErrorCode.SCHEMA
assert schema_outcome.attempts == 1


## Checkpoint Export and Verification

Chapter metadata names Chapter 1 as the base Checkpoint. Export carries every unchanged source and cumulative test forward, applies this chapter's replacements, writes a deterministic full-file manifest, then compiles, installs without network or dependency resolution, imports, and runs all tests before publishing Chapter 2.

In [ ]:
PYPROJECT_SOURCE = r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "agent-harness"
version = "0.2.0"
description = "Chapter 2 async-first streaming agent runtime"
readme = "README.md"
requires-python = ">=3.11"
dependencies = ["openai>=1.40,<3"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
agent_harness = ["py.typed"]

[tool.pytest.ini_options]
testpaths = ["tests"]
'''


In [ ]:
README_SOURCE = r'''# Agent Harness — Chapter 2 Checkpoint

This cumulative Checkpoint adds one async-first `AgentRuntime` to the Chapter 1 model seam. `start(...)` returns an `AgentRunHandle` with an ordered passive Event stream, cancellation, and an eventual typed `AssistantOutcome`; `run(...)` is its completion-oriented convenience.

The Runtime owns classified retries and bounded backoff. Its default retry schedule is 2, 4, and 8 seconds, and a provider `Retry-After` hint is honored only through 60 seconds. Failures and cancellation after acceptance settle into provider-neutral Assistant outcomes; invalid input still fails before acceptance. Failed partial attempts remain diagnostic Events rather than conversation history, and aggregate Run guards are opt-in.

All ordinary tests are deterministic and offline. The inherited real-endpoint smoke test remains supplementary and runs only when `AGENT_HARNESS_REAL_SMOKE=1` and all three explicit endpoint variables are supplied.
'''


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '02_async_runtime.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch02',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '02_async_runtime.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch02',
) == ()
checkpoint_result


## Public API Summary

Construct `AgentRuntime(adapter, model, retry_policy=..., sleeper=..., run_guard=...)`. Call `start(messages)` to receive an `AgentRunHandle` with `events()`, `result()`, and `cancel()`, or await `run(messages)` directly. The terminal `AssistantOutcome` exposes the Assistant message, provider-neutral Stop Reason, Usage, normalized Model Error, and attempt count. Runtime performs no terminal I/O and imposes no aggregate RunGuard by default.